In [2]:
#!git clone -b anni https://github.com/ruicatzzz/aigc-detector.git
%pip install -r requirements.txt -q


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%cd src
!export HF_TOKEN="YOUR_HUGGINGFACE_TOKEN"
!python download_sid_set.py --out_dir ../data_sid/train --n_per_class 1000 --split train
!python download_sid_set.py --out_dir ../data_sid/test --n_per_class 500 --split validation
!python build_test_transforms.py --test_dir ../data_sid/test --out_dir ../data_sid/test_transformed

/Users/anni/Desktop/aigc-detector/src
Streaming saberzl/SID_Set split='train' ...
Resolving data files: 100%|█████████████████| 34/34 [00:00<00:00, 39811.93it/s]
  progress: REAL=63 FAKE=137
  progress: REAL=133 FAKE=267
  progress: REAL=207 FAKE=393
  progress: REAL=272 FAKE=528
  progress: REAL=337 FAKE=663
  progress: REAL=409 FAKE=791
  progress: REAL=477 FAKE=923
  progress: REAL=600 FAKE=1000
  progress: REAL=800 FAKE=1000
  progress: REAL=1000 FAKE=1000
Done. Saved REAL=1000 FAKE=1000 to ../data_sid/train/
Metadata (incl. real/full_synthetic/tampered breakdown) written to ../data_sid/train/train_metadata.csv
^C
Streaming saberzl/SID_Set split='validation' ...
Resolving data files: 100%|█████████████████| 34/34 [00:00<00:00, 39253.05it/s]
  progress: REAL=68 FAKE=132
  progress: REAL=144 FAKE=256
  progress: REAL=210 FAKE=390
  progress: REAL=300 FAKE=500
  progress: REAL=500 FAKE=500
Done. Saved REAL=500 FAKE=500 to ../data_sid/test/
Metadata (incl. real/full_synthetic/tampered 

In [ ]:
!python train.py --data_dir ../data_sid --epochs 5 --run_name resnet50_baseline_v2
!python train.py --data_dir ../data_sid --epochs 5 --augment --run_name resnet50_augmented_v2


Using device: cpu
Loaded 6000 training samples (augment=False)

model.safetensors: downloading bytes:   0% 0.00/102M [00:00<?, ?B/s]
model.safetensors: downloading bytes:   6% 5.99M/102M [00:00<00:14, 6.87MB/s]
model.safetensors: downloading bytes:  15% 15.0M/102M [00:00<00:04, 18.6MB/s,  590kB/s  ]
model.safetensors: downloading bytes:  30% 30.3M/102M [00:01<00:01, 37.4MB/s, 2.19MB/s  ]
model.safetensors: downloading bytes:  38% 39.2M/102M [00:01<00:01, 48.4MB/s, 2.92MB/s  ]
model.safetensors: reconstructing file:  41% 41.8M/102M [00:01<00:01, 55.8MB/s, 2.79MB/s  ]
model.safetensors: downloading bytes:  57% 58.0M/102M [00:01<00:00, 65.4MB/s, 4.28MB/s  ]
model.safetensors: downloading bytes:  72% 74.0M/102M [00:01<00:00, 70.3MB/s, 5.95MB/s  ]
model.safetensors: downloading bytes:  84% 86.0M/102M [00:01<00:00, 80.0MB/s, 6.93MB/s  ]
model.safetensors: downloading bytes: 100% 96.2M/96.2M [00:02<00:00, 46.3MB/s, 8.92MB/s  ]
model.safetensors: reconstructing file: 100% 102M/102M [00:02<00:0

In [ ]:
!python evaluate.py --checkpoint ../checkpoints/resnet50_baseline_v2.pt \
                     --data_dir ../data_sid --transformed_dir ../data_sid/test_transformed \
                     --out ../outputs/baseline_v2_results.csv

!python evaluate.py --checkpoint ../checkpoints/resnet50_augmented_v2.pt \
                     --data_dir ../data_sid --transformed_dir ../data_sid/test_transformed \
                     --out ../outputs/augmented_v2_results.csv

In [ ]:
#Comparing between baseline and augmented
import pandas as pd

baseline = pd.read_csv("../outputs/baseline_v2_results.csv")
augmented = pd.read_csv("../outputs/augmented_v2_results.csv")

comparison = baseline[["condition", "accuracy"]].merge(
    augmented[["condition", "accuracy"]],
    on="condition",
    suffixes=("_baseline", "_augmented")
)
comparison["delta"] = comparison["accuracy_augmented"] - comparison["accuracy_baseline"]
comparison = comparison.sort_values("delta", ascending=False)
comparison.style.background_gradient(subset=["delta"], cmap="RdYlGn", vmin=-0.05, vmax=0.05) \
                .format({"accuracy_baseline": "{:.3f}", "accuracy_augmented": "{:.3f}", "delta": "{:+.3f}"})
print(f"Baseline avg accuracy:  {baseline['accuracy'].mean():.3f}")
print(f"Augmented avg accuracy: {augmented['accuracy'].mean():.3f}")
print(f"Average lift: {(augmented['accuracy'].mean() - baseline['accuracy'].mean()):+.3f}")
print(f"Baseline spread (max-min): {baseline['accuracy'].max() - baseline['accuracy'].min():.3f}")
print(f"Augmented spread (max-min): {augmented['accuracy'].max() - augmented['accuracy'].min():.3f}")

print(comparison.to_string())




In [ ]:
#Importing CIFAKE dataset to do a cross evaluation
!mkdir -p ~/.kaggle && echo YOUR_KAGGLE_TOKEN > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
!unzip -q cifake-real-and-ai-generated-synthetic-images.zip -d ../data_cifake

In [ ]:
!ls ../data_cifake
!ls ../data_cifake/test  # confirm REAL/FAKE folders exist

In [ ]:
#build transformed set with cifake data, limit 500 for faster runtime
%cd
!python build_test_transforms.py --test_dir ../data_cifake/test --out_dir ../data_cifake/test_transformed --limit 500

In [ ]:
#run SID model against CIFAKE data
!python evaluate.py --checkpoint ../checkpoints/resnet50_augmented_combined.pt \
                    --backbone resnet50 \
                    --data_dir ../data_cifake \
                    --transformed_dir ../data_cifake/test_transformed \
                    --out ../outputs/sid_model_on_cifake.csv

In [ ]:
#compare CIFAKE against SID evaluation
import pandas as pd

on_sid = pd.read_csv("../outputs/augmented_combined_results.csv")
on_cifake = pd.read_csv("../outputs/sid_model_on_cifake.csv")

print(f"SID-trained model on SID test set:    {on_sid['accuracy'].mean():.4f}")
print(f"SID-trained model on CIFAKE test set: {on_cifake['accuracy'].mean():.4f}")

gap = on_sid['accuracy'].mean() - on_cifake['accuracy'].mean()
print(f"Generalization gap: {gap:+.4f}")

In [ ]:
#dont run yet


!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"
%cd /content/aigc-detector
!git add outputs/ checkpoints/
!git commit -m "v2 training on Colab GPU"
!git push origin <your-branch>